# 🚀 Część 3: Uvicorn, Gunicorn i inne serwery

**Cel:** Zrozumieć czym są konkretne serwery (Uvicorn, Gunicorn), jak się mają do WSGI/ASGI i jakie są alternatywy.

**Analogia przewodnia:** 🗣️ Standard języka vs konkretny tłumacz

---

## 1. Standard vs Implementacja - kluczowa różnica

### Analogia: Język vs Tłumacz

```
Język angielski = STANDARD komunikacji
  ↓ Kto mówi po angielsku?
  - Tłumacz A (John)
  - Tłumacz B (Mary)
  - Tłumacz C (Tom)
```

Wszyscy mówią **tym samym językiem** (angielski), ale są **różnymi osobami**.

---

### W kontekście Python web:

```
WSGI = STANDARD (interfejs/protokół)
  ↓ Kto implementuje WSGI?
  - Gunicorn      ⭐
  - uWSGI
  - mod_wsgi
  - Waitress

ASGI = STANDARD (interfejs/protokół)
  ↓ Kto implementuje ASGI?
  - Uvicorn       ⭐
  - Daphne
  - Hypercorn
```

**KLUCZOWE:**
- **WSGI/ASGI** = standard ("jak mówić")
- **Gunicorn/Uvicorn** = programy implementujące standard ("kto mówi")

---

## 2. Uvicorn - co to dokładnie jest?

### Czym jest Uvicorn?

**Uvicorn** = **ASGI server** (implementacja standardu ASGI)

```
Uvicorn to PROGRAM, który:
1. Nasłuchuje na porcie (np. 8000)
2. Odbiera HTTP requesty
3. Przekształca je zgodnie ze standardem ASGI:
   → wywołuje: await app(scope, receive, send)
4. Odbiera response z aplikacji
5. Przekształca z powrotem na HTTP response
```

### Uvicorn vs ASGI:

```
ASGI = standard/protokół (abstrakcja)
Uvicorn = konkretna implementacja tego standardu (kod)
```

**Analogia:**
- ASGI = przepis na robienie pizzy (standard)
- Uvicorn = konkretny kucharz, który robi pizzę według tego przepisu

---

### Dlaczego Uvicorn jest popularny?

✅ **Bardzo szybki** - napisany w C (uvloop) + Cython
✅ **Prosty w użyciu** - jedna komenda
✅ **Domyślny dla FastAPI** - FastAPI docs używają Uvicorn
✅ **Production-ready** - używany w produkcji przez wiele firm
✅ **Aktywny development** - ciągle rozwijany

---

### Jak używać Uvicorn?

```bash
# Instalacja
pip install uvicorn[standard]

# Development - auto-reload przy zmianach
uvicorn main:app --reload

# Production - 4 worker processes
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4

# Z logami
uvicorn main:app --log-level info
```

### Co oznaczają parametry:

- `main:app` - moduł `main.py`, zmienna `app`
- `--reload` - auto-reload przy zmianach (TYLKO DEV!)
- `--host 0.0.0.0` - słuchaj na wszystkich interfejsach (nie tylko localhost)
- `--port 8000` - port
- `--workers 4` - ile procesów worker (CPU cores × 2 + 1)

---

## 3. Gunicorn - co to dokładnie jest?

### Czym jest Gunicorn?

**Gunicorn** = **WSGI server** (implementacja standardu WSGI)

**Green Unicorn** - "zielony jednorożec"

```
Gunicorn to PROGRAM, który:
1. Nasłuchuje na porcie (np. 8000)
2. Odbiera HTTP requesty
3. Przekształca je zgodnie ze standardem WSGI:
   → wywołuje: app(environ, start_response)
4. Odbiera response z aplikacji
5. Przekształca z powrotem na HTTP response
```

### Gunicorn vs WSGI:

```
WSGI = standard/protokół
Gunicorn = konkretna implementacja tego standardu
```

---

### Dlaczego Gunicorn jest popularny?

✅ **Stabilny** - istnieje od 2010, dojrzały
✅ **Prosty** - łatwa konfiguracja
✅ **Pre-fork worker model** - wiele procesów worker
✅ **Domyślny dla Django** - Django docs używają Gunicorn
✅ **Production-ready** - używany przez Spotify, Instagram, etc.

---

### Jak używać Gunicorn?

```bash
# Instalacja
pip install gunicorn

# Django
gunicorn myproject.wsgi:application --workers 4

# Flask
gunicorn app:app --workers 4 --bind 0.0.0.0:8000

# Z więcej opcji
gunicorn myproject.wsgi:application \
  --workers 4 \
  --bind 0.0.0.0:8000 \
  --log-level info \
  --access-logfile -
```

---

## 4. Gunicorn + Uvicorn Workers - HYBRID!

### Problem:

- **Gunicorn** = świetny process manager, ale tylko WSGI (sync)
- **Uvicorn** = świetny ASGI server, ale słaby process management

### Rozwiązanie: Użyj Gunicorn jako managera, a Uvicorn jako workera!

```bash
# Gunicorn z Uvicorn workers (ASYNC!)
gunicorn main:app \
  --workers 4 \
  --worker-class uvicorn.workers.UvicornWorker \
  --bind 0.0.0.0:8000
```

### Jak to działa:

```
Gunicorn (master process)
    ↓ zarządza:
    ├─ Uvicorn Worker 1 (process 1) - obsługuje async requesty
    ├─ Uvicorn Worker 2 (process 2) - obsługuje async requesty
    ├─ Uvicorn Worker 3 (process 3) - obsługuje async requesty
    └─ Uvicorn Worker 4 (process 4) - obsługuje async requesty
```

**Korzyści:**
✅ **Gunicorn:** Graceful restarts, process monitoring, logging
✅ **Uvicorn:** Async handling, szybkość, ASGI features

**To jest często używane w produkcji dla FastAPI!**

---

## 5. Django Development Server - co to?

### Django `runserver`:

```bash
python manage.py runserver
```

**Co to jest?**
- **Prosty WSGI server wbudowany w Django**
- Klasa: `django.core.servers.basehttp.WSGIServer`
- Oparty na `wsgiref.simple_server` z Python standard library

### Cechy:

✅ **Auto-reload** - restartuje się przy zmianach w kodzie
✅ **Łatwy setup** - zero konfiguracji
✅ **Wbudowany** - nie trzeba nic instalować

❌ **Jednowątkowy** - obsługuje tylko 1 request naraz
❌ **Wolny** - nie zoptymalizowany
❌ **Niebezpieczny** - brak security features
❌ **TYLKO DEV!** - Django docs mówi: "DON'T use this in production"

---

### Django w produkcji:

```bash
# NIE używaj runserver!
# python manage.py runserver  # ❌ TYLKO DEV!

# Używaj Gunicorn:
gunicorn myproject.wsgi:application --workers 4  # ✅ PRODUCTION
```

---

### Django + ASGI (od wersji 3.0):

Django od 3.0 obsługuje **zarówno WSGI jak i ASGI**!

```python
# myproject/wsgi.py (tradycyjne)
from django.core.wsgi import get_wsgi_application
application = get_wsgi_application()

# myproject/asgi.py (nowe!)
from django.core.asgi import get_asgi_application
application = get_asgi_application()
```

**Production ASGI:**
```bash
# Uvicorn
uvicorn myproject.asgi:application --workers 4

# Daphne (starszy ASGI server dla Django)
daphne myproject.asgi:application
```

---

## 6. Flask Development Server - co to?

### Flask `flask run`:

```bash
flask run
# lub
python app.py
```

**Co to jest?**
- **Werkzeug WSGI server** (wbudowany w Flask)
- Prosty development server

### Cechy:

✅ **Auto-reload** - przy zmianach
✅ **Debugger** - interaktywny debugger w przeglądarce
✅ **Prosty** - zero konfiguracji

❌ **Wolny**
❌ **Jednowątkowy** (domyślnie)
❌ **TYLKO DEV!** - Flask docs: "Do not use the development server in production"

---

### Flask w produkcji:

```bash
# NIE używaj flask run!
# flask run  # ❌ TYLKO DEV!

# Używaj Gunicorn:
gunicorn app:app --workers 4  # ✅ PRODUCTION
```

---

## 7. Alternatywy dla Uvicorn (ASGI servers)

### 1. Daphne

**Od twórców Django Channels** (WebSocket support dla Django)

```bash
pip install daphne
daphne myproject.asgi:application
```

**Pros:**
✅ Pierwszy ASGI server (2016)
✅ Dobra integracja z Django
✅ Stabilny

**Cons:**
❌ Wolniejszy niż Uvicorn
❌ Mniej popularny obecnie

---

### 2. Hypercorn

**Od twórców Quart** (async Flask alternative)

```bash
pip install hypercorn
hypercorn main:app --workers 4
```

**Pros:**
✅ Obsługuje HTTP/2 i HTTP/3!
✅ Bardzo nowoczesny
✅ Aktywny development

**Cons:**
❌ Mniej popularny niż Uvicorn
❌ Wolniejszy niż Uvicorn (trochę)

---

### Porównanie ASGI servers:

| Server | Wydajność | HTTP/2 | HTTP/3 | Popularność | Use case |
|--------|-----------|--------|--------|-------------|----------|
| **Uvicorn** | ⭐⭐⭐⭐⭐ | ❌ | ❌ | ⭐⭐⭐⭐⭐ | FastAPI (domyślny) |
| **Daphne** | ⭐⭐⭐ | ❌ | ❌ | ⭐⭐⭐ | Django Channels |
| **Hypercorn** | ⭐⭐⭐⭐ | ✅ | ✅ | ⭐⭐ | HTTP/2, HTTP/3 apps |

**Rekomendacja:** Uvicorn dla większości projektów FastAPI.

---

## 8. Alternatywy dla Gunicorn (WSGI servers)

### 1. uWSGI

**Bardzo wydajny, ale skomplikowany**

```bash
pip install uwsgi
uwsgi --http :8000 --wsgi-file app.py --callable app --processes 4
```

**Pros:**
✅ Bardzo szybki
✅ Wiele opcji konfiguracji
✅ Integracja z nginx (uwsgi protocol)

**Cons:**
❌ Skomplikowana konfiguracja
❌ Trudniejszy w użyciu niż Gunicorn

---

### 2. mod_wsgi (dla Apache)

**Moduł Apache HTTP Server**

```apache
# Apache config
WSGIDaemonProcess myapp python-path=/path/to/myapp
WSGIProcessGroup myapp
WSGIScriptAlias / /path/to/myapp/wsgi.py
```

**Pros:**
✅ Integracja z Apache
✅ Dobra dla legacy systemów

**Cons:**
❌ Apache jest wolniejszy niż nginx
❌ Bardziej skomplikowany

---

### 3. Waitress

**Pure-Python WSGI server**

```bash
pip install waitress
waitress-serve --port=8000 app:app
```

**Pros:**
✅ Działa na Windows (Gunicorn nie!)
✅ Prosty
✅ Pure Python (łatwa instalacja)

**Cons:**
❌ Wolniejszy niż Gunicorn
❌ Mniej features

---

### Porównanie WSGI servers:

| Server | Wydajność | Łatwość użycia | Windows | Popularność | Use case |
|--------|-----------|----------------|---------|-------------|----------|
| **Gunicorn** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ❌ | ⭐⭐⭐⭐⭐ | Django/Flask (domyślny) |
| **uWSGI** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ✅ | ⭐⭐⭐⭐ | High-performance, nginx |
| **mod_wsgi** | ⭐⭐⭐ | ⭐⭐ | ✅ | ⭐⭐ | Apache environments |
| **Waitress** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ✅ | ⭐⭐ | Windows, prototypy |

**Rekomendacja:** Gunicorn dla większości projektów Django/Flask.

---

## 9. Workers - ile potrzeba?

### Formula dla liczby workers:

```
    (2 × liczba CPU cores) + 1
```

### Przykłady:

| CPU cores | Workers | Komenda |
|-----------|---------|----------|
| 1 core | 3 | `--workers 3` |
| 2 cores | 5 | `--workers 5` |
| 4 cores | 9 | `--workers 9` |
| 8 cores | 17 | `--workers 17` |

### Dlaczego ta formula?

- **2×** - pokrywa I/O wait (gdy worker czeka na DB/network)
- **+1** - zapas

### W praktyce:

```bash
# Sprawdź ile masz CPU cores
python -c "import os; print(os.cpu_count())"
# Output: 4

# Użyj 9 workers (2 × 4 + 1)
gunicorn app:app --workers 9
```

**UWAGA:** Nie przesadź! Więcej workers = więcej RAM.

---

## 🎯 Podsumowanie: Standard vs Implementacja

### Hierarchia:

```
┌─────────────────────────────────────────────┐
│        STANDARD (interfejs/protokół)        │
├─────────────────────────────────────────────┤
│  WSGI (sync)          │  ASGI (async)       │
└──────┬────────────────┴──────┬──────────────┘
       │                       │
       ↓ implementacje         ↓ implementacje
┌──────────────────┐    ┌──────────────────┐
│ Gunicorn ⭐      │    │ Uvicorn ⭐       │
│ uWSGI            │    │ Daphne           │
│ mod_wsgi         │    │ Hypercorn        │
│ Waitress         │    │                  │
└──────────────────┘    └──────────────────┘
```

---

### Wybór serwera dla Twojego projektu:

#### FastAPI (async):
```bash
# Opcja 1: Tylko Uvicorn (prostota)
uvicorn main:app --workers 4

# Opcja 2: Gunicorn + Uvicorn workers (recommended dla production)
gunicorn main:app --workers 4 --worker-class uvicorn.workers.UvicornWorker
```

#### Django (sync):
```bash
# Development
python manage.py runserver  # Wbudowany server

# Production
gunicorn myproject.wsgi:application --workers 4
```

#### Django (async - od 3.0):
```bash
# Production z ASGI
uvicorn myproject.asgi:application --workers 4
# lub
daphne myproject.asgi:application
```

#### Flask (sync):
```bash
# Development
flask run  # Werkzeug server

# Production
gunicorn app:app --workers 4
```

---

## 🎯 Kluczowe wnioski:

1. **WSGI/ASGI** = standard (abstrakcja)
2. **Gunicorn/Uvicorn** = konkretne programy implementujące standard
3. **Uvicorn** = najpopularniejszy ASGI server (FastAPI)
4. **Gunicorn** = najpopularniejszy WSGI server (Django/Flask)
5. **Gunicorn + Uvicorn workers** = hybrid (process management + async)
6. **Django/Flask development servers** = TYLKO DEV, nigdy production!
7. **Workers formula:** (2 × CPU cores) + 1

---

## 📚 Dalej:

**Następny notebook:** `04_nginx_reverse_proxy.ipynb`
- Czym jest nginx?
- Czym jest reverse proxy?
- Jak nginx się różni od Uvicorn/Gunicorn?
- Po co nginx PRZED Uvicorn?
- Alternatywy dla nginx

---